# 00A — Large Language Models (LLMs) and Vision–Language Models (VLMs) in PyTorch

## What these terms mean

A **Large Language Model (LLM)** is a neural network that assigns probabilities to text tokens: small pieces of text represented by integer IDs. A **Vision–Language Model (VLM)** also accepts visual information alongside text. Qwen3.5 contains language and vision components; a text-only run need not supply an image.

A **parameter** is a numerical value adjusted during training. **Weights** are the learned parameter tensors, including matrices; a **tensor** is a multidimensional array. An **architecture** specifies operations and tensor shapes. A **configuration** selects their dimensions. A **checkpoint** stores parameter values and configuration; an inference checkpoint may lack the extra state needed to resume training.

**Pre-training** learns broad patterns from a corpus. **Fine-tuning** continues updating an existing model for a domain, behavior, or task. A **forward pass** computes outputs with the current weights. A **loss** measures a training mismatch; **backpropagation** computes gradients describing how parameter changes affect that loss. An **optimizer** uses gradients to update selected parameters.

**What problem does this solve?** Pretrained weights provide a learned starting point. Constructing the same architecture with random weights gives matching shapes, not matching knowledge. A forward pass alone does not train the model.


## Three independent experiment choices

| Choice | Question | Examples |
|---|---|---|
| Training objective | What evidence should make the model change? | Domain text, demonstrated answers, preferred responses |
| Parameter-update strategy | Which numbers may change? | Full tuning, selective tuning, low-rank adapters |
| Feedback source | Where does the evidence come from? | Written text, human judgments, model judgments, programmatic checks |

**Full fine-tuning** updates all parameters. **Selective fine-tuning** updates an explicitly chosen subset. **Low-Rank Adaptation (LoRA)** freezes original matrices and learns small added matrices. **Quantized Low-Rank Adaptation (QLoRA)** also stores the frozen base in reduced precision; this course uses 4-bit base storage and floating-point adapter computation. Quantization changes numerical representation, not the number of layers.

An **adapter** contains the added trainable parameters and any explicitly saved extra modules. It generally needs its matching base checkpoint. **Supervised Fine-Tuning (SFT)** means learning demonstrated answers: “SFT with LoRA” combines an objective and a parameter strategy.


## NovaBot example and a hand calculation

**Fictional scenario:** NovaBot's manual says, “NovaBot has three modes: idle, mapping, navigation.” A language model receives a question about those modes. A vision–language model can also receive a picture of its status panel. This is an explanatory scenario, not a measured output of the tiny model.

For a linear layer, LoRA uses
$$W_{\mathrm{effective}}=W_0+\frac{\alpha}{r}BA.$$
Here $W_0$ is the frozen original matrix; $A$ and $B$ are trainable matrices; $r$ is their rank; and $\alpha$ sets the update scale. If $W_0$ is $8\times8$ and $r=2$, $A$ is $2\times8$ and $B$ is $8\times2$. We train 32 added entries instead of 64 original entries. The original entries still exist; this is not a total GPU-memory estimate.

A forward pass produces **logits**, unnormalized scores. **Softmax** converts them to probabilities summing to one. Hand-constructed logits $[\log(0.8),\log(0.2)]$ give probabilities $[0.8,0.2]$, where $\log$ is the natural logarithm. Generation chooses tokens from such distributions; learning changes the distributions through gradients.


## Connection to this notebook

MODEL_PATH identifies checkpoint files. The from_pretrained() call constructs modules and loads weights; model.config describes dimensions, and model.state_dict() maps names to tensors. named_modules() lists computational blocks. **Forward hooks** record their output shapes.

The shape [batch, tokens, hidden] describes groups of token representations; the output head changes the last dimension to vocabulary size. requires_grad marks parameters eligible for gradients, while parameter_report() counts frozen and trainable values.

The vision encoder turns image patches into features, and the merger maps them into the language representation. Lesson 07 runs that branch. The actual tiny fixture has random weights and a synthetic vocabulary: it tests mechanics, not knowledge of NovaBot.


## Common confusions and quick check

A config describes structure, not learned knowledge. An adapter is not the complete base model.

1. Does one call to model(**batch) change the weights?
2. Can supervised answer training use either full tuning or LoRA?

<details>
<summary>Answers</summary>

1. No. Training also needs a loss, backward computation, and an optimizer update.
2. Yes. The objective specifies the learning signal; the update strategy specifies which parameters respond to it.

</details>


## Before running the experiment

**Learning goals:** follow the official forward path, inspect tensor dimensions, distinguish architecture from weights, and identify the parameters used by each tuning strategy.

**Prerequisites:** basic Python. The concepts needed for this lesson are introduced above. Run every cell in order in a fresh kernel. The default `tiny_cpu` mode has random weights and a synthetic vocabulary: output quality is not evidence of Qwen's capabilities. Use `local_pretrained` for an already downloaded checkpoint.

[Course index](README.md) · [Execution and data flow](../docs/EXECUTION_AND_DATA_FLOW.md)

**Experiment contract:** inspect inputs before training, keep held-out records separate, check gradients/parameter changes, then save and reload. Each notebook is independent.

[Terminology reference](../docs/GLOSSARY.md) · [Compare training methods](../docs/TRAINING_METHODS.md)


## Choose the experiment

The model directory must contain its own weights, tokenizer, processor and chat template. The environment variables below are optional; edit the parameter cell directly in Jupyter. Files created by this lesson stay under its output directory.

For local weights, set `FTLAB_DEVICE` to `cpu`, `mps`, `cuda`, or `auto` before launching. CPU/MPS default to ordinary LoRA; CUDA retains its QLoRA profile. Restart the kernel when changing devices after a Trainer has initialized. Selecting a device does not guarantee the full experiment fits its memory.


In [ ]:
LESSON = "00a"
# Parameters: change these before running the notebook from top to bottom.
import copy
import csv
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

from finetunelab.devices import (
    activate_runtime,
    resolve_runtime,
)
from finetunelab.education import (
    inspect_local_checkpoint,
    make_tiny_checkpoint,
    project_root,
    token_table,
)
from finetunelab.tuning import parameter_report

MODE = os.environ.get("FTLAB_NOTEBOOK_MODE", "tiny_cpu")
LOCAL_MODEL_PATH = Path(os.environ.get("FTLAB_LOCAL_MODEL", "models/Qwen3.5-2B"))
LOCAL_TEACHER_PATH = Path(os.environ.get("FTLAB_LOCAL_TEACHER", "models/Qwen3.5-4B"))
ROOT = project_root()
DATA_ROOT = Path(os.environ.get("FTLAB_LESSON_DATA", str(ROOT / "examples/education")))
OUTPUT_ROOT = Path(os.environ.get("FTLAB_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebooks")))
OUTPUT = OUTPUT_ROOT / LESSON
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
assert MODE in {"tiny_cpu", "local_pretrained"}
# tiny_cpu remains an offline CPU fixture; real checkpoints use the selected backend.
REQUESTED_DEVICE = "cpu" if MODE == "tiny_cpu" else os.environ.get("FTLAB_DEVICE", "auto")
RUNTIME = resolve_runtime(device=REQUESTED_DEVICE, dtype=os.environ.get("FTLAB_DTYPE"))
activate_runtime(RUNTIME)
DEVICE = torch.device(RUNTIME.device)
DTYPE = RUNTIME.torch_dtype
ATTENTION = "eager" if MODE == "tiny_cpu" else RUNTIME.attention
print(RUNTIME.report())
MODEL_PATH = (
    make_tiny_checkpoint(OUTPUT / "initial", seed=SEED)
    if MODE == "tiny_cpu"
    else LOCAL_MODEL_PATH.expanduser().resolve()
)
checkpoint_info = inspect_local_checkpoint(MODEL_PATH)
print({"mode": MODE, "device": str(DEVICE), "checkpoint": str(MODEL_PATH)})
print(checkpoint_info["files"])
if DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

## Load the local checkpoint

`from_pretrained()` constructs the official PyTorch modules and fills their tensors from the checkpoint. `local_files_only=True` prevents a missing local file from becoming a network download. BF16 and NF4 serve different roles: computation precision versus storage of the frozen base.


In [ ]:
# A checkpoint includes both weights and the preprocessing contract.
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
load_kwargs = {
    "local_files_only": True,
    "dtype": DTYPE,
    "device_map": {"": str(DEVICE)},
    "attn_implementation": ATTENTION,
}
# Quantization is a separate choice. CPU/MPS use unquantized LoRA.
_default_qlora = (
    MODE == "local_pretrained" and DEVICE.type == "cuda" and LESSON not in {"00a", "00b", "04"}
)
USE_QLORA = os.environ.get("FTLAB_USE_QLORA", str(_default_qlora)).lower() in {"1", "true", "yes"}
if USE_QLORA and DEVICE.type != "cuda":
    raise ValueError("This CPU/MPS profile supports ordinary LoRA; set FTLAB_USE_QLORA=false.")
if USE_QLORA:
    from transformers import BitsAndBytesConfig

    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    load_kwargs["device_map"] = {"": torch.cuda.current_device()}
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
if not USE_QLORA:
    model.to(DEVICE)
model.config.use_cache = False
print(type(model).__name__, parameter_report(model))

## Read the implementation that actually runs

The loaded object is a `torch.nn.Module`. Inspecting its class shows exactly which installed implementation runs. `config.json` sets dimensions; `state_dict` supplies learned tensors. The tiny fixture retains a complete 3:1 language-layer cycle. Real 2B/4B dimensions come from their checkpoint configs, not the tiny defaults.


In [ ]:
import inspect

from transformers.models.qwen3_5 import modeling_qwen3_5

text_config = model.config.text_config
print(
    {
        "language_layers": text_config.num_hidden_layers,
        "layer_types": text_config.layer_types,
        "hidden_size": text_config.hidden_size,
        "attention_heads": text_config.num_attention_heads,
        "kv_heads": text_config.num_key_value_heads,
        "vision_depth": model.config.vision_config.depth,
    }
)
print("Installed source:", inspect.getfile(type(model)))
print(inspect.getsource(modeling_qwen3_5.Qwen3_5DecoderLayer.forward))
display(
    [
        (name, type(module).__name__)
        for name, module in model.named_modules()
        if name.endswith(
            ("embed_tokens", "linear_attn", "self_attn", "mlp", "norm", "lm_head", "merger")
        )
    ]
)

## Trace a real forward pass

Embeddings produce `[batch, tokens, hidden]`. Decoder blocks preserve that shape while mixing information. The LM head produces `[batch, tokens, vocabulary]`. Gated DeltaNet carries recurrent state rather than materializing the same attention matrix as full attention. Hook only representative modules and remove hooks afterwards.


In [ ]:
shapes = {}
handles = []


def shape_hook(name):
    def record(module, args, output):
        value = output[0] if isinstance(output, tuple) else output
        if isinstance(value, torch.Tensor):
            shapes[name] = tuple(value.shape)

    return record


for name, module in model.named_modules():
    if name.endswith(("embed_tokens", "layers.0", "layers.3", "lm_head")):
        handles.append(module.register_forward_hook(shape_hook(name)))
tokens = tokenizer("Water freezes at zero degrees .", return_tensors="pt")
tokens.pop("token_type_ids", None)
with torch.no_grad():
    outputs = model(**{k: v.to(DEVICE) for k, v in tokens.items()}, use_cache=False)
for handle in handles:
    handle.remove()
display(shapes)
print("Logits:", tuple(outputs.logits.shape))
print("Representative weight:", next(iter(model.state_dict().items()))[0])

## Architecture versus checkpoint values

Constructing a model from config initializes weights. Loading from a directory restores weights. For a real 2B/4B model, use the meta device to inspect structure without allocating another full copy. Never treat newly initialized tensors as downloaded pretrained weights.


In [ ]:
if MODE == "tiny_cpu":
    from transformers import Qwen3_5ForConditionalGeneration

    random_model = Qwen3_5ForConditionalGeneration(copy.deepcopy(model.config))
    key = next(iter(model.state_dict()))
    assert model.state_dict()[key].shape == random_model.state_dict()[key].shape
    assert not torch.equal(model.state_dict()[key], random_model.state_dict()[key])
    del random_model
else:
    with torch.device("meta"):
        structure_only = AutoModelForImageTextToText.from_config(model.config)
    print("Structure-only parameters:", sum(p.numel() for p in structure_only.parameters()))
    del structure_only
print("Weight files:", [f for f in checkpoint_info["files"] if "safetensors" in f])

## From local Q&A to train/validation/test records

A dataset row is not yet a tensor. Preserve the original group identity so examples from one conversation stay together. These tiny held-out splits demonstrate plumbing; use representative, larger splits in real experiments.


In [ ]:
# Convert local Q&A rows to canonical conversations; preserve provenance.
QA_FILE = Path(os.environ.get("FTLAB_QA_FILE", str(DATA_ROOT / "qa.csv")))
if QA_FILE.suffix.lower() == ".csv":
    with QA_FILE.open(encoding="utf-8", newline="") as handle:
        raw_rows = list(csv.DictReader(handle))
elif QA_FILE.suffix.lower() == ".jsonl":
    raw_rows = [
        json.loads(line)
        for line in QA_FILE.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    raise ValueError("This converter accepts CSV or JSONL Q&A files.")
for row in raw_rows:
    if (
        not row.get("group_id")
        or not row.get("question", "").strip()
        or not row.get("answer", "").strip()
    ):
        raise ValueError("Every Q&A needs a group_id, nonempty question, and nonempty answer.")
    row.setdefault("rejected", "")
records = [
    {
        "group_id": row["group_id"],
        "messages": [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        "prompt": row["question"],
        "chosen": row["answer"],
        "rejected": row["rejected"],
    }
    for row in raw_rows
]


def split_records(rows, seed=SEED):
    # Deduplicate before splitting. Never split one document/conversation group.
    unique = {}
    for row in rows:
        key = json.dumps(row["messages"], sort_keys=True, ensure_ascii=False)
        unique.setdefault(key, row)
    groups = sorted({row["group_id"] for row in unique.values()})
    if len(groups) < 3:
        raise ValueError("Provide at least three independent document/conversation groups.")
    random.Random(seed).shuffle(groups)
    validation_groups, test_groups = set(groups[:1]), set(groups[1:2])
    splits = {"train": [], "validation": [], "test": []}
    for row in unique.values():
        split = (
            "validation"
            if row["group_id"] in validation_groups
            else "test"
            if row["group_id"] in test_groups
            else "train"
        )
        splits[split].append(row)
    return splits


splits = split_records(records)
for split, rows in splits.items():
    with (OUTPUT / f"{split}.jsonl").open("w", encoding="utf-8") as handle:
        for row in rows:
            canonical = {"group_id": row["group_id"], "messages": row["messages"]}
            handle.write(json.dumps(canonical, ensure_ascii=False) + "\n")
print({split: len(rows) for split, rows in splits.items()})
print("Raw:", raw_rows[0])
print("Canonical:", records[0])

## Chat rendering, tokenization and loss masking

The chat template supplies role delimiters. Attention masks describe real positions versus padding; labels choose prediction targets. A user token can be visible to attention while its label is `-100`. Assistant EOS should remain supervised even if its ID equals the padding ID.


In [ ]:
import re


def render(messages, generation=False):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=generation,
        enable_thinking=False,
    )


def encode_conversation(messages):
    # Template-provided generation masks are preferred. They include assistant EOS.
    template = tokenizer.chat_template or ""
    if re.search(r"{%-?\s*generation\s*-?%}", template):
        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            enable_thinking=False,
        )
        ids = encoded["input_ids"]
        supervised = encoded["assistant_masks"]
    else:
        # For templates without generation annotations, verify prefix alignment.
        # Do not guess a token count by separately tokenizing the answer.
        ids = tokenizer(render(messages), add_special_tokens=False)["input_ids"]
        supervised = [0] * len(ids)
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            prefix = tokenizer(render(messages[:index], generation=True), add_special_tokens=False)[
                "input_ids"
            ]
            completed = tokenizer(render(messages[: index + 1]), add_special_tokens=False)[
                "input_ids"
            ]
            if ids[: len(prefix)] != prefix or ids[: len(completed)] != completed:
                raise ValueError(
                    "Template is not prefix-stable; use a training template with generation tags."
                )
            supervised[len(prefix) : len(completed)] = [1] * (len(completed) - len(prefix))
    if not any(supervised[1:]):
        raise ValueError("No assistant target tokens remain.")
    return {
        "input_ids": ids,
        "labels": [t if keep else -100 for t, keep in zip(ids, supervised, strict=False)],
    }


def collate_text(rows):
    items = [encode_conversation(row["messages"]) for row in rows]
    encoded = tokenizer.pad(
        [{"input_ids": item["input_ids"]} for item in items],
        padding=True,
        return_tensors="pt",
    )
    # Padding labels are independent of the pad token ID (pad may equal EOS).
    labels = torch.full_like(encoded["input_ids"], -100)
    for index, item in enumerate(items):
        labels[index, : len(item["labels"])] = torch.tensor(item["labels"])
    encoded["labels"] = labels
    return dict(encoded)


batch = collate_text(splits["train"][:2])
print(render(splits["train"][0]["messages"]))
print({name: tuple(value.shape) for name, value in batch.items()})
display(token_table(tokenizer, batch))

## Select parameters that may change

LoRA freezes the pretrained matrices and adds low-rank updates. Its zero-initialized B matrices mean some A gradients may be zero on the first step; at least one adapter must change. The frozen vision backbone is excluded. Set `TUNING` to `full` or `selective` to compare on the tiny model; full tuning of a real model needs substantially more memory.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# This is the same choice represented by tuning.strategy in a framework YAML.
TUNING = "lora"
if USE_QLORA and TUNING != "lora":
    raise ValueError("Full/selective tuning requires reloading with USE_QLORA=False.")
if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()
if TUNING == "lora":
    model = get_peft_model(
        model,
        LoraConfig(
            r=int(os.environ.get("FTLAB_LORA_RANK", "4" if MODE == "tiny_cpu" else "16")),
            lora_alpha=8 if MODE == "tiny_cpu" else 32,
            target_modules="all-linear",
            exclude_modules=r".*(?:visual|vision).*",
            task_type="CAUSAL_LM",
            lora_dropout=0.0,
        ),
    )
elif TUNING == "selective":
    for name, parameter in model.named_parameters():
        parameter.requires_grad = ("norm" in name or "lm_head" in name) and "visual" not in name
elif TUNING != "full":
    raise ValueError(TUNING)
if MODE == "local_pretrained" and not USE_QLORA:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
trainable = [p for p in model.parameters() if p.requires_grad]
print(parameter_report(model))
# A small parameter sample avoids copying a 2B/4B model just to audit updates.
before = {name: p.detach().flatten()[:32].cpu().clone() for name, p in model.named_parameters()}
frozen_names = {name for name, p in model.named_parameters() if not p.requires_grad}

## Vision and tuning map

Image patches pass through the vision transformer and merger before replacing visual placeholder embeddings in the language sequence. Lesson 07 runs this branch and inspects `pixel_values` and `image_grid_thw`.

| Strategy | Updated tensors | Stored artifact |
|---|---|---|
| Full | All parameters, including vision | Complete checkpoint |
| Selective | Chosen layers/norms/head | Complete checkpoint |
| LoRA | Low-rank language adapters; optional merger | Adapter plus explicitly saved modules |
| QLoRA | Adapters over a quantized frozen base | Adapter, not a new 4-bit base |

**Exercises:** find the first full-attention layer; compare language and vision parameter counts; explain why the LM head changes the last dimension. For the 4B local snapshot, repeat inspection before attempting training.


In [ ]:
(OUTPUT / "lesson_report.json").write_text(
    json.dumps(
        {"mode": MODE, "forward_shapes": shapes, "parameters": parameter_report(model)}, indent=2
    ),
    encoding="utf-8",
)